# Harmonization

First nesting approach — maps the USA import/export distribution onto the state sectors. This notebook covers the **harmonization** step (config + function + analysis of the harmonized tables). Reloads aggregated tables from disk; writes harmonized tables back.

# Librairies

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

# Harmonization

#### Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════════
# Batch harmonization — apply harmonize_windc_to_oecd to all available years
# ════════════════════════════════════════════════════════════════════════════════════
import sys
from pathlib import Path
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, str(ROOT / "pipeline"))
from harmonize import harmonize_windc_to_oecd, load_oecd_targets, OECD_ROOT, FD_CATS

# ── Single knob: which aggregated WiNDC build to harmonize ────────────────
# Points at grav_fric_<WINDC_VERSION>_aggregated -> grav_fric_<WINDC_VERSION>_harmonized
WINDC_VERSION = "v3.1_RAS"

IOT_USA   = ROOT / "data/interim/IOT/IOT_USA"
WINDC_AGG = IOT_USA / f"grav_fric_{WINDC_VERSION}_aggregated"
OUT_DIR   = IOT_USA / f"grav_fric_{WINDC_VERSION}_harmonized"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("WINDC_VERSION:", WINDC_VERSION)
print("WINDC_AGG    :", WINDC_AGG.name)
print("OUT_DIR      :", OUT_DIR.name)

#### Function

In [ ]:
# ── Discover available years ────────────────────────────────────────────────
def windc_years():
    return sorted(int(p.stem.split("_")[1]) for p in WINDC_AGG.glob("IOT_*.npz")
                   if p.stem.split("_")[1].isdigit())

def find_oecd_file(year):
    for folder in OECD_ROOT.iterdir():
        if not folder.is_dir():
            continue
        for f in folder.glob(f"{year}_*.parquet"):
            return f
    return None

available_years = [y for y in windc_years() if find_oecd_file(y) is not None]
print(f"Years available in both WinDC and OECD: {available_years}")

# ── Resolve common sectors once (same across years) ────────────────────────
d0 = np.load(next(WINDC_AGG.glob("IOT_*.npz")), allow_pickle=True)
secs_wd = list(dict.fromkeys(str(s) for s in d0["proposed_sectors"]))
dfo0 = pd.read_parquet(find_oecd_file(available_years[0]))
secs_oe_all = [c.split("_", 1)[1] for c in dfo0.columns
                if c.startswith("USA_") and c.split("_", 1)[1] not in FD_CATS]
secs_common = [s for s in secs_wd if s in set(secs_oe_all)]
wd_idx = [secs_wd.index(s)     for s in secs_common]
oe_idx = [secs_oe_all.index(s) for s in secs_common]
print(f"Common sectors: {len(secs_common)}")

# ── Loop over years ────────────────────────────────────────────────────────
summary_rows = []
for k, year in enumerate(available_years, 1):
    print(f"[{k}/{len(available_years)}] {year} — ", end="", flush=True)
    oecd_t = load_oecd_targets(year, secs_common, secs_oe_all, oe_idx)
    npz_wd = np.load(WINDC_AGG / f"IOT_{year}.npz", allow_pickle=True)
    out, trans = harmonize_windc_to_oecd(npz_wd, oecd_t, wd_idx, secs_common)

    # Save harmonized table — all blocks incl. WiNDC tax blocks (kept as-is)
    np.savez_compressed(OUT_DIR / f"IOT_{year}_harmonized.npz", **out)
    np.savez_compressed(OUT_DIR / f"transform_{year}.npz",
                        sf_Z=trans["sf_Z"], sf_VA=trans["sf_VA"], sf_EX=trans["sf_EX"],
                        sf_M=trans["sf_M"], sf_F=trans["sf_F"],
                        secs_common=np.array(secs_common))

    # Verify aggregate match (Z/F/VA/EX/M matched; taxes intentionally kept)
    n_r, n_sc = len(out["regions"]), len(secs_common)
    Z_h  = out["Z"].reshape(n_r, n_sc, n_r, n_sc).sum(axis=(0, 2))
    F_h  = out["F"].reshape(n_r, n_sc, -1).sum(axis=(0, 2))
    VA_h = out["VA"].reshape(n_r, n_sc).sum(0)
    EX_h = out["EX"].reshape(n_r, n_sc).sum(0)
    M_h  = out["M_interm"].reshape(n_r, n_sc).sum(0)
    TX_h = out["taxes"].reshape(n_r, n_sc).sum(0)
    max_err = max(
        np.abs(Z_h.sum(1) - oecd_t["Z"].sum(1)).max(),
        np.abs(Z_h.sum(0) - oecd_t["Z"].sum(0)).max(),
        np.abs(VA_h - oecd_t["VA"]).max(), np.abs(EX_h - oecd_t["EX"]).max(),
        np.abs(M_h  - oecd_t["M"]).max(),  np.abs(F_h  - oecd_t["F"]).max())
    summary_rows.append(dict(
        year=year, ras_iters=trans["ras_info"]["iters"], max_agg_err=max_err,
        TX_windc_Bn=TX_h.sum(), TX_oecd_Bn=oecd_t["TX"].sum(),
        status="ok" if max_err < 1e-5 else "imbalanced"))
    print(f"RAS {trans['ras_info']['iters']} it, max agg err = {max_err:.2e}, "
          f"TX kept = {TX_h.sum():.1f} Bn$ (OECD TLS {oecd_t['TX'].sum():.1f})")

df_sum = pd.DataFrame(summary_rows)
print(f"\n══════ BATCH SUMMARY ══════")
print(df_sum.to_string(index=False))
df_sum.to_csv(OUT_DIR / "harmonization_summary.csv", index=False)
print(f"\nFiles saved to: {OUT_DIR}")


#### Analysis of the harmonized tables

Before (raw aggregated WiNDC) vs After (harmonized) vs OECD USA target — year 2017.
Z, F, VA, EX, M are matched to the OECD US block; the WiNDC tax blocks are kept
as-is (shown for reference, not harmonized).

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Harmonization improvement — BEFORE (raw aggregated) vs AFTER (harmonized)
#                             vs OECD USA target  (year 2017)
# ════════════════════════════════════════════════════════════════════════════
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LogNorm

YEAR = 2017
from paths import ROOT
WINDC_VERSION = globals().get("WINDC_VERSION", "v3.1_RAS")   # set in the config cell
WINDC_RAW  = ROOT / f"data/interim/IOT/IOT_USA/grav_fric_{WINDC_VERSION}_aggregated"
WINDC_HARM = ROOT / f"data/interim/IOT/IOT_USA/grav_fric_{WINDC_VERSION}_harmonized"
OECD_AGG   = ROOT / "data/interim/IOT/OCDE ICIO aggregated"
FIG_DIR    = ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
FD_CATS     = ["DPABR", "GFCF", "GGFC", "HFCE", "INVNT", "NPISH"]
WINDC_SCALE = 1000.0   # Bn$ -> M$ (OECD is in M$)
EPS_REF     = 10.0     # M$ floor for relative diffs

def find_oecd_file(year):
    for folder in OECD_AGG.iterdir():
        if folder.is_dir():
            for f in folder.glob(f"{year}_*.parquet"):
                return f
    raise FileNotFoundError(year)

# ── Aggregate a WiNDC npz (state-level) to the USA block, in M$ ─────────────
def agg_blocks(npz, sect_order):
    secs = [str(s) for s in npz["proposed_sectors"]]
    nr   = npz["Z"].shape[0] // len(secs)
    idx  = [secs.index(s) for s in sect_order]
    Z4 = npz["Z"].reshape(nr, len(secs), nr, len(secs))
    Z  = Z4[:, idx, :, :][:, :, :, idx].sum((0, 2)) * WINDC_SCALE
    F  = npz["F"].reshape(nr, len(secs), -1)[:, idx, :].sum((0, 2)) * WINDC_SCALE
    one = lambda k: npz[k].reshape(nr, len(secs))[:, idx].sum(0) * WINDC_SCALE
    return dict(Z=Z, F=F, VA=one("VA"), EX=one("EX"),
                M=one("M_interm"), TX=one("taxes"))

# ── Load after (harmonized), before (raw aggregated), OECD target ──────────
npz_h   = np.load(WINDC_HARM / f"IOT_{YEAR}_harmonized.npz", allow_pickle=True)
sectors = [str(s) for s in npz_h["proposed_sectors"]]
n_sc    = len(sectors)
after   = agg_blocks(npz_h, sectors)

npz_r   = np.load(WINDC_RAW / f"IOT_{YEAR}.npz", allow_pickle=True)
before  = agg_blocks(npz_r, sectors)

dfo = pd.read_parquet(find_oecd_file(YEAR))
usa_rows = [f"USA_{s}" for s in sectors]
usa_fds  = [f"USA_{fd}" for fd in FD_CATS if f"USA_{fd}" in dfo.columns]
world_sec_cols = [c for c in dfo.columns if "_" in c and len(c.split("_")[0]) == 3
                  and not c.startswith("USA_") and c.split("_", 1)[1] not in FD_CATS]
world_fd_cols  = [c for c in dfo.columns if "_" in c and len(c.split("_")[0]) == 3
                  and not c.startswith("USA_") and c.split("_", 1)[1] in FD_CATS]
world_rows     = [r for r in dfo.index if "_" in r and len(r.split("_")[0]) == 3
                  and not r.startswith("USA_")]
oecd = dict(
    Z  = dfo.loc[usa_rows, usa_rows].values.astype(float),
    F  = dfo.loc[usa_rows, usa_fds].values.astype(float).sum(1),
    VA = dfo.loc["VA",  usa_rows].values.astype(float),
    TX = dfo.loc["TLS", usa_rows].values.astype(float),
    M  = dfo.loc[world_rows, usa_rows].values.astype(float).sum(0),
    EX = (dfo.loc[usa_rows, world_sec_cols].values.astype(float).sum(1)
          + dfo.loc[usa_rows, world_fd_cols].values.astype(float).sum(1)),
)

# ── Master before/after divergence table ───────────────────────────────────
def div(h, o):  return np.linalg.norm(h - o) / (np.linalg.norm(o) + 1e-12)
def rmse(h, o): return np.sqrt(((h - o) ** 2).mean())

rows = []
for blk in ["Z", "F", "VA", "EX", "M", "TX"]:
    rows.append(dict(
        block=blk if blk != "TX" else "TX (kept)",
        OECD=oecd[blk].sum(), before=before[blk].sum(), after=after[blk].sum(),
        div_before=div(before[blk], oecd[blk]), div_after=div(after[blk], oecd[blk]),
        rmse_before=rmse(before[blk], oecd[blk]), rmse_after=rmse(after[blk], oecd[blk])))
tbl = pd.DataFrame(rows)
tbl["div_reduction"] = 1 - tbl["div_after"] / tbl["div_before"].replace(0, np.nan)
print(f"══ Harmonization improvement — year {YEAR}  (totals in M$) ══")
print(tbl.to_string(index=False, formatters={
    "OECD": "{:,.0f}".format, "before": "{:,.0f}".format, "after": "{:,.0f}".format,
    "div_before": "{:.4f}".format, "div_after": "{:.4f}".format,
    "rmse_before": "{:,.0f}".format, "rmse_after": "{:,.0f}".format,
    "div_reduction": "{:+.1%}".format}))
tbl.to_csv(FIG_DIR / f"harmonization_before_after_{YEAR}.csv", index=False)


In [ ]:
# ── Bar chart: divergence to OECD, before vs after, per matched block ───────
plot_blocks = ["Z", "F", "VA", "EX", "M"]
db = [div(before[b], oecd[b]) for b in plot_blocks]
da = [div(after[b],  oecd[b]) for b in plot_blocks]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(plot_blocks)); w = 0.38
ax.bar(x - w/2, db, w, label="before (raw aggregated)", color="#bd5e2c")
ax.bar(x + w/2, da, w, label="after (harmonized)",      color="#3577a8")
for i, (b, a) in enumerate(zip(db, da)):
    ax.text(i - w/2, b, f"{b:.2f}",  ha="center", va="bottom", fontsize=8)
    ax.text(i + w/2, a, f"{a:.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(plot_blocks)
ax.set_ylabel("relative Frobenius divergence to OECD")
ax.set_title(f"Divergence WiNDC ↔ OECD USA — before vs after harmonization ({YEAR})",
             fontweight="bold")
ax.legend(frameon=False); ax.grid(axis="y", ls=":", alpha=.4)
plt.tight_layout()
plt.savefig(FIG_DIR / f"divergence_before_after_{YEAR}.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
# ── Z heatmaps: OECD vs before vs after + relative diffs ───────────────────
Zb, Za, Zo = before["Z"], after["Z"], oecd["Z"]

def reldiff(h, o):
    return np.where(np.abs(o) >= EPS_REF, (h - o) / (np.abs(o) + 1e-12), np.nan)

rb, ra = reldiff(Zb, Zo), reldiff(Za, Zo)
eps = 1.0
lnorm = LogNorm(vmin=eps, vmax=max(Zo.max(), Zb.max(), Za.max()))
both  = np.concatenate([rb[~np.isnan(rb)], ra[~np.isnan(ra)]])
p95   = np.nanpercentile(np.abs(both), 95)
dnorm = TwoSlopeNorm(vmin=-p95, vcenter=0, vmax=p95)
labs  = [s[:14] for s in sectors]

fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes[1, 0].axis("off")
specs = [
    (0, 0, np.clip(Zo, eps, None), "OECD USA",                 "YlOrRd", lnorm, "M$"),
    (0, 1, np.clip(Zb, eps, None), "WiNDC before (raw)",       "YlOrRd", lnorm, "M$"),
    (0, 2, np.clip(Za, eps, None), "WiNDC after (harmonized)", "YlOrRd", lnorm, "M$"),
    (1, 1, rb, f"rel diff BEFORE   div={div(Zb, Zo):.3f}",     "RdBu_r", dnorm, "ratio"),
    (1, 2, ra, f"rel diff AFTER    div={div(Za, Zo):.3f}",     "RdBu_r", dnorm, "ratio"),
]
for r, c, mat, title, cmap, norm, unit in specs:
    ax = axes[r, c]
    im = ax.imshow(mat, cmap=cmap, norm=norm, aspect="auto")
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_xticks(range(n_sc)); ax.set_xticklabels(labs, rotation=60, ha="right", fontsize=5)
    ax.set_yticks(range(n_sc)); ax.set_yticklabels(labs, fontsize=5)
    cb = plt.colorbar(im, ax=ax, shrink=.75, pad=.02); cb.ax.tick_params(labelsize=6)
    if unit == "ratio":
        cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:+.0%}"))
fig.suptitle(f"Z[USA × USA] intermediate flows — harmonization effect ({YEAR})",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR / f"Z_before_after_{YEAR}.png", dpi=160, bbox_inches="tight")
plt.show()


Remark:

The 0s in the windc table are a big problem: 176 cells (12.9%), whereas oecd has none, and these same cells in oecd carry $51Bn. These 0 cells can never be lifted by the RAS which is multiplicative.

We must test a seeding of the null cells before RAS as an optional parameter and compare the results

#### Approach 2 — zero-cell seeding before RAS

WiNDC's aggregated Z has **176 zero cells (12.9%)** where OECD is positive
(~51 Bn$ of OECD mass). RAS is multiplicative, so it can never fill a structural
zero: the missing margin is forced onto the other cells of the same row/column,
distorting the fit. Approach 2 seeds those zeros with a small flat floor
(`seed_zeros` × mean positive cell) **before** the RAS, then spreads each seeded
aggregate cell uniformly across state pairs. F/VA/EX/M/taxes are identical to
approach 1 — only Z changes.

In [ ]:
# ── Approach 2: seed zero cells before RAS — frac sweep + batch save ────────
import importlib, harmonize
importlib.reload(harmonize)   # reload so the seed_zeros param is picked up in a warm kernel
from harmonize import harmonize_windc_to_oecd, load_oecd_targets, OECD_ROOT, FD_CATS
# WINDC_AGG / WINDC_VERSION come from the config cell (not re-imported, to keep the override)
WINDC_VERSION = globals().get("WINDC_VERSION", "v3.1_RAS")

OUT_SEED = ROOT / f"data/interim/IOT/IOT_USA/grav_fric_{WINDC_VERSION}_harmonized_seeded"
OUT_SEED.mkdir(parents=True, exist_ok=True)

# Resolve common sectors + OECD targets (same setup as approach 1)
d0 = np.load(next(WINDC_AGG.glob("IOT_*.npz")), allow_pickle=True)
secs_wd = list(dict.fromkeys(str(s) for s in d0["proposed_sectors"]))
secs_oe_all = [c.split("_", 1)[1] for c in pd.read_parquet(find_oecd_file(YEAR)).columns
               if c.startswith("USA_") and c.split("_", 1)[1] not in FD_CATS]
secs_common = [s for s in secs_wd if s in set(secs_oe_all)]
wd_idx = [secs_wd.index(s)     for s in secs_common]
oe_idx = [secs_oe_all.index(s) for s in secs_common]
oecd_t = load_oecd_targets(YEAR, secs_common, secs_oe_all, oe_idx)
npz_wd = np.load(WINDC_AGG / f"IOT_{YEAR}.npz", allow_pickle=True)

def _divZ(out):
    nr = len(out["regions"]); ns = len(secs_common)
    Zh = out["Z"].reshape(nr, ns, nr, ns).sum((0, 2))
    return np.linalg.norm(Zh - oecd_t["Z"]) / np.linalg.norm(oecd_t["Z"])

import numpy as np, pandas as pd

# Raw aggregated WiNDC Z (for the masks) + OECD cell-structure target
regions_wd       = list(dict.fromkeys(str(r) for r in npz_wd["regions"]))
n_r, n_sw, n_sc  = len(regions_wd), len(secs_wd), len(secs_common)
Z_us_raw = (npz_wd["Z"].reshape(n_r, n_sw, n_r, n_sw)[:, wd_idx][:, :, :, wd_idx]).sum((0, 2))
O        = oecd_t["Z"]
eps      = 1e-9

seed_m = (Z_us_raw <= eps) & (O > eps)   # WiNDC=0, OECD>0  -> cells seeding is meant to fill
off_m  =  O <= eps                       # OECD=0           -> any mass here is spurious
pos_m  =  Z_us_raw > eps                 # originally real WiNDC cells

_Zagg = lambda out: out["Z"].reshape(n_r, n_sc, n_r, n_sc).sum((0, 2))
div   = lambda a, b: np.linalg.norm(a - b) / np.linalg.norm(b)

# Baseline (no seed) to measure how much seeding distorts the *real* cells
out0, _ = harmonize_windc_to_oecd(npz_wd, oecd_t, wd_idx, secs_common, seed_zeros=0.0)
Zh0     = _Zagg(out0)

rows = []
for frac in [0.0, 1e-3, 5e-3, 1e-2, 2e-2, 5e-2, 1e-1]:
    out, tr = harmonize_windc_to_oecd(npz_wd, oecd_t, wd_idx, secs_common, seed_zeros=frac)
    Zh = _Zagg(out)
    rows.append(dict(
        frac      = frac,
        divZ      = div(Zh, O),                                   # global fit (lower = better)
        div_seed  = div(Zh[seed_m], O[seed_m]),                   # fit on the cells we lift
        corr      = np.corrcoef(Zh.ravel(), O.ravel())[0, 1],     # cell-structure agreement
        fill_seed = Zh[seed_m].sum() / O[seed_m].sum(),           # ~1 = right mass on lifted cells
        offsupp   = Zh[off_m].sum() / 1000,                       # Bn$ dumped where OECD=0  (lower better)
        drift_pos = np.abs(Zh[pos_m] - Zh0[pos_m]).sum() / Zh0[pos_m].sum(),  # distortion of real cells
        ras_err   = tr["ras_info"]["err"],                        # RAS convergence
        ras_iters = tr["ras_info"]["iters"],
    ))
df = pd.DataFrame(rows)
print("seed_zeros sweep — multi-indicator:")
print(df.to_string(index=False, float_format=lambda v: f"{v:.4g}"))


In [ ]:

# Chosen background-seed level (best of the sweep), saved for downstream use
SEED_FRAC = 1e-3  # sweep optimum
out, tr = harmonize_windc_to_oecd(npz_wd, oecd_t, wd_idx, secs_common, seed_zeros=SEED_FRAC)
np.savez_compressed(OUT_SEED / f"IOT_{YEAR}_harmonized.npz", **out)
print(f"\nApproach 2 (seed_zeros={SEED_FRAC}) saved to {OUT_SEED.name}: "
      f"{tr['n_seeded']} cells lifted, divZ={_divZ(out):.4f}")


In [ ]:
# ── Compare approach 1 (no seed) vs approach 2 (seeded) — Z block ───────────
npz_a1 = np.load(WINDC_HARM / f"IOT_{YEAR}_harmonized.npz", allow_pickle=True)
npz_a2 = np.load(OUT_SEED   / f"IOT_{YEAR}_harmonized.npz", allow_pickle=True)
Z_a1 = agg_blocks(npz_a1, sectors)["Z"]
Z_a2 = agg_blocks(npz_a2, sectors)["Z"]
Zb, Zo = before["Z"], oecd["Z"]
seeded = (Zb <= 0) & (Zo > 0)   # cells WiNDC=0 but OECD>0

cmp = pd.DataFrame([
    dict(variant="before (raw)",          divZ=div(Zb,   Zo), rmseZ=np.sqrt(((Zb  - Zo)**2).mean())),
    dict(variant="approach 1 (no seed)",  divZ=div(Z_a1, Zo), rmseZ=np.sqrt(((Z_a1 - Zo)**2).mean())),
    dict(variant="approach 2 (seeded)",   divZ=div(Z_a2, Zo), rmseZ=np.sqrt(((Z_a2 - Zo)**2).mean())),
])
print(f"Zero cells lifted: {int(seeded.sum())} ({seeded.mean()*100:.1f}%), "
      f"OECD mass there = {Zo[seeded].sum()/1000:.1f} Bn$")
print(f"  approach 1 places {Z_a1[seeded].sum()/1000:7.2f} Bn$ in those cells (forced to ~0)")
print(f"  approach 2 places {Z_a2[seeded].sum()/1000:7.2f} Bn$ in those cells")
print(cmp.to_string(index=False, formatters={"divZ": "{:.4f}".format, "rmseZ": "{:,.0f}".format}))

# Bar chart
fig, ax = plt.subplots(figsize=(7, 4.5))
labels = ["before\n(raw)", "approach 1\n(no seed)", "approach 2\n(seeded)"]
vals   = [div(Zb, Zo), div(Z_a1, Zo), div(Z_a2, Zo)]
bars   = ax.bar(labels, vals, color=["#bd5e2c", "#3577a8", "#2ca25f"])
for r, v in zip(bars, vals):
    ax.text(r.get_x() + r.get_width()/2, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("divergence of Z to OECD")
ax.set_title(f"Z harmonization — zero-cell seeding effect ({YEAR})", fontweight="bold")
ax.grid(axis="y", ls=":", alpha=.4)
plt.tight_layout()
plt.savefig(FIG_DIR / f"Z_seeding_compare_{YEAR}.png", dpi=180, bbox_inches="tight")
plt.show()

# Heatmaps: rel-diff approach 1 vs approach 2 + seeded-cell map
def reldiff(h, o):
    return np.where(np.abs(o) >= EPS_REF, (h - o) / (np.abs(o) + 1e-12), np.nan)
r1, r2 = reldiff(Z_a1, Zo), reldiff(Z_a2, Zo)
both = np.concatenate([r1[~np.isnan(r1)], r2[~np.isnan(r2)]])
p95  = np.nanpercentile(np.abs(both), 95)        # kept for info / possible print
CAP  = 1.0                                        # ±100%  (rel-diff of 1.0)
dn   = TwoSlopeNorm(vmin=-CAP, vcenter=0, vmax=CAP)
labs = [s[:14] for s in sectors]


fig, axes = plt.subplots(1, 3, figsize=(21, 7))
specs = [
    (r1, f"rel diff approach 1   div={div(Z_a1, Zo):.3f}", "RdBu_r", dn,   "pct"),
    (r2, f"rel diff approach 2   div={div(Z_a2, Zo):.3f}", "RdBu_r", dn,   "pct"),
    (seeded.astype(float), f"seeded cells ({int(seeded.sum())})  WiNDC=0, OECD>0",
     "Greys", None, None),
]
for ax, (mat, title, cmap, norm, fmt) in zip(axes, specs):
    im = ax.imshow(mat, cmap=cmap, norm=norm, aspect="auto")
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_xticks(range(n_sc)); ax.set_xticklabels(labs, rotation=60, ha="right", fontsize=5)
    ax.set_yticks(range(n_sc)); ax.set_yticklabels(labs, fontsize=5)
    cb = plt.colorbar(im, ax=ax, shrink=.75, pad=.02); cb.ax.tick_params(labelsize=6)
    if fmt == "pct":
        cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:+.0%}"))
fig.suptitle(f"Approach 1 vs 2 — Z rel-diff to OECD + seeded-cell map ({YEAR})",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR / f"Z_seeding_heatmaps_{YEAR}.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
seeds = [0.0, 1e-3, 2e-3, 3e-3, 3.3e-3, 3.5e-3, 4e-3, 1e-2, 1e-1]
for i in range(len(seeds)):
    SEED_FRAC = seeds[i]  # sweep optimum
    out, tr = harmonize_windc_to_oecd(npz_wd, oecd_t, wd_idx, secs_common, seed_zeros=SEED_FRAC)
    np.savez_compressed(OUT_SEED / f"IOT_{YEAR}_harmonized.npz", **out)
    print(f"\nApproach 2 (seed_zeros={SEED_FRAC}) saved to {OUT_SEED.name}: "
        f"{tr['n_seeded']} cells lifted, divZ={_divZ(out):.4f}")
    
    # ── Compare approach 1 (no seed) vs approach 2 (seeded) — Z block ───────────
    npz_a1 = np.load(WINDC_HARM / f"IOT_{YEAR}_harmonized.npz", allow_pickle=True)
    npz_a2 = np.load(OUT_SEED   / f"IOT_{YEAR}_harmonized.npz", allow_pickle=True)
    Z_a1 = agg_blocks(npz_a1, sectors)["Z"]
    Z_a2 = agg_blocks(npz_a2, sectors)["Z"]
    Zb, Zo = before["Z"], oecd["Z"]
    seeded = (Zb <= 0) & (Zo > 0)   # cells WiNDC=0 but OECD>0

    cmp = pd.DataFrame([
        dict(variant="before (raw)",          divZ=div(Zb,   Zo), rmseZ=np.sqrt(((Zb  - Zo)**2).mean())),
        dict(variant="approach 1 (no seed)",  divZ=div(Z_a1, Zo), rmseZ=np.sqrt(((Z_a1 - Zo)**2).mean())),
        dict(variant="approach 2 (seeded)",   divZ=div(Z_a2, Zo), rmseZ=np.sqrt(((Z_a2 - Zo)**2).mean())),
    ])
    print(f"Zero cells lifted: {int(seeded.sum())} ({seeded.mean()*100:.1f}%), "
        f"OECD mass there = {Zo[seeded].sum()/1000:.1f} Bn$")
    print(f"  approach 1 places {Z_a1[seeded].sum()/1000:7.2f} Bn$ in those cells (forced to ~0)")
    print(f"  approach 2 places {Z_a2[seeded].sum()/1000:7.2f} Bn$ in those cells")
    print(cmp.to_string(index=False, formatters={"divZ": "{:.4f}".format, "rmseZ": "{:,.0f}".format}))

    # Bar chart
    fig, ax = plt.subplots(figsize=(7, 4.5))
    labels = ["before\n(raw)", "approach 1\n(no seed)", "approach 2\n(seeded)"]
    vals   = [div(Zb, Zo), div(Z_a1, Zo), div(Z_a2, Zo)]
    bars   = ax.bar(labels, vals, color=["#bd5e2c", "#3577a8", "#2ca25f"])
    for r, v in zip(bars, vals):
        ax.text(r.get_x() + r.get_width()/2, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("divergence of Z to OECD")
    ax.set_title(f"Z harmonization — zero-cell seeding effect ({YEAR})", fontweight="bold")
    ax.grid(axis="y", ls=":", alpha=.4)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"Z_seeding_compare_{YEAR}.png", dpi=180, bbox_inches="tight")
    plt.show()

    #Bar chart : divergence of Z to OECD for seeded cells only
    fig, ax = plt.subplots(figsize=(7, 4.5))
    labels = ["approach 1\n(no seed)", "approach 2\n(seeded)"]
    vals   = [div(Z_a1[seeded], Zo[seeded]), div(Z_a2[seeded], Zo[seeded])]
    bars   = ax.bar(labels, vals, color=["#3577a8", "#2ca25f"])
    for r, v in zip(bars, vals):    
        ax.text(r.get_x() + r.get_width()/2, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)   
    ax.set_ylabel("divergence of Z to OECD (seeded cells only)")
    ax.set_title(f"Z harmonization — zero-cell seeding effect ({YEAR})", fontweight="bold")
    ax.grid(axis="y", ls=":", alpha=.4)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"Z_seeding_compare_seeded_only_{YEAR}.png", dpi=180, bbox_inches="tight")
    plt.show()

    # Heatmaps: rel-diff approach 1 vs approach 2 + seeded-cell map
    def reldiff(h, o):
        return np.where(np.abs(o) >= EPS_REF, (h - o) / (np.abs(o) + 1e-12), np.nan)
    r1, r2 = reldiff(Z_a1, Zo), reldiff(Z_a2, Zo)
    both = np.concatenate([r1[~np.isnan(r1)], r2[~np.isnan(r2)]])
    p95  = np.nanpercentile(np.abs(both), 95)        # kept for info / possible print
    CAP  = 1.0                                        # ±100%  (rel-diff of 1.0)
    dn   = TwoSlopeNorm(vmin=-CAP, vcenter=0, vmax=CAP)
    labs = [s[:14] for s in sectors]


    fig, axes = plt.subplots(1, 3, figsize=(21, 7))
    specs = [
        (r1, f"rel diff approach 1   div={div(Z_a1, Zo):.3f}", "RdBu_r", dn,   "pct"),
        (r2, f"rel diff approach 2   div={div(Z_a2, Zo):.3f}", "RdBu_r", dn,   "pct"),
        (seeded.astype(float), f"seeded cells ({int(seeded.sum())})  WiNDC=0, OECD>0",
        "Greys", None, None),
    ]
    for ax, (mat, title, cmap, norm, fmt) in zip(axes, specs):
        im = ax.imshow(mat, cmap=cmap, norm=norm, aspect="auto")
        ax.set_title(title, fontsize=9, fontweight="bold")
        ax.set_xticks(range(n_sc)); ax.set_xticklabels(labs, rotation=60, ha="right", fontsize=5)
        ax.set_yticks(range(n_sc)); ax.set_yticklabels(labs, fontsize=5)
        cb = plt.colorbar(im, ax=ax, shrink=.75, pad=.02); cb.ax.tick_params(labelsize=6)
        if fmt == "pct":
            cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:+.0%}"))
    fig.suptitle(f"Approach 1 vs 2 — Z rel-diff to OECD + seeded-cell map ({YEAR})",
                fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"Z_seeding_heatmaps_{YEAR}.png", dpi=160, bbox_inches="tight")
    plt.show()


In [ ]:
# ── Divergence vs seed level: whole Z table vs seeded cells only ─────────────
# For each seed in `seeds`, harmonize with that floor and record two divergences
# to OECD:
#   - whole Z table  (full USA×USA intermediate block)
#   - seeded cells only (the WiNDC=0 / OECD>0 cells the seeding is meant to fill)
seeds = [0.0, 1e-3, 2e-3, 3e-3, 3.5e-3, 4e-3, 5e-3, 6e-3, 1e-2, 1e-1]
div_total, div_seeded = [], []
for frac in seeds[:-1]:
    out, _ = harmonize_windc_to_oecd(npz_wd, oecd_t, wd_idx, secs_common, seed_zeros=frac)
    Zh = _Zagg(out)
    div_total.append(div(Zh, O))
    div_seeded.append(div(Zh[seed_m], O[seed_m]))

# Two very different scales (whole table ≈ 0.16, seeded cells ≈ 1–5) -> twin axes
fig, ax1 = plt.subplots(figsize=(8, 5))
x = np.arange(len(seeds[:-1]))
l1, = ax1.plot(x, div_total, "o-", color="#3577a8", label="whole Z table")
ax1.set_xlabel("seed_zeros  (× mean positive cell)")
ax1.set_ylabel("divergence of whole Z to OECD", color="#3577a8")
ax1.tick_params(axis="y", labelcolor="#3577a8")
ax1.set_xticks(x); ax1.set_xticklabels([f"{s:g}" for s in seeds[:-1]], rotation=45, ha="right")

ax2 = ax1.twinx()
l2, = ax2.plot(x, div_seeded, "s--", color="#2ca25f", label="seeded cells only")
ax2.set_ylabel("divergence of seeded cells to OECD", color="#2ca25f")
ax2.tick_params(axis="y", labelcolor="#2ca25f")


ax1.set_title(f"Divergence vs seed level — whole table vs seeded cells ({YEAR})",
              fontweight="bold")
ax1.legend(handles=[l1, l2], frameon=False, loc="center right")
ax1.grid(axis="y", ls=":", alpha=.4)
plt.tight_layout()
plt.savefig(FIG_DIR / f"divergence_vs_seed_{YEAR}.png", dpi=180, bbox_inches="tight")
plt.show()


Conclusion : 

Seeding with 3e-3 has no negative effect on global Z table but improves divergence for seeded cells (cells that are 0 for WiNDC but not for OECD)


In [ ]:
SEED_FRAC = 3e-3  # sweep optimum
out, tr = harmonize_windc_to_oecd(npz_wd, oecd_t, wd_idx, secs_common, seed_zeros=SEED_FRAC)
np.savez_compressed(OUT_SEED / f"IOT_{YEAR}_harmonized.npz", **out)
print(f"\nApproach 2 (seed_zeros={SEED_FRAC}) saved to {OUT_SEED.name}: "
      f"{tr['n_seeded']} cells lifted, divZ={_divZ(out):.4f}")

## Final analysis

In [ ]:
Za = agg_blocks(np.load(OUT_SEED / f"IOT_{YEAR}_harmonized.npz", allow_pickle=True), sectors)["Z"]

In [ ]:
# ── Z heatmaps — abs diff / rel% / row-normalised / total-normalised / ratio ──
#    (rows: WiNDC before vs after harmonization;  all vs OECD USA, in M$)
short   = [s[:13] for s in sectors]
tick_kw = dict(rotation=60, ha="right", fontsize=6)

def draw_hm(ax, mat, title, cmap, vmin, vmax, vcenter=None, fmt="%"):
    if vcenter is not None:
        im = ax.imshow(mat, cmap=cmap, norm=TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax), aspect="auto")
    else:
        im = ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(n_sc)); ax.set_xticklabels(short, **tick_kw)
    ax.set_yticks(range(n_sc)); ax.set_yticklabels(short, fontsize=6)
    ax.set_title(title, fontsize=8, fontweight="bold")
    ax.set_xlabel("Buying sector", fontsize=7); ax.set_ylabel("Selling sector", fontsize=7)
    cb = plt.colorbar(im, ax=ax, shrink=0.75, pad=0.02); cb.ax.tick_params(labelsize=7)
    if   fmt == "%":  cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    elif fmt == "x":  cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1f}×"))
    elif fmt == "M$": cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

ref     = np.abs(Zo) >= EPS_REF                                       # mask tiny OECD cells
adiff   = lambda Z: np.where(ref, np.abs(Z - Zo), np.nan)             # |Z − OECD|  (M$)
reldp   = lambda Z: np.where(ref, np.abs(Z - Zo) / (np.abs(Zo) + 1e-12) * 100, np.nan)  # |rel| %
rownorm = lambda Z: (Z - Zo) / (Z.sum(1)[:, None] + 1e-12) * 100      # (Z−OECD)/row-sum(Z)   signed %
totnorm = lambda Z: (Z - Zo) / (Z.sum()        + 1e-12) * 100         # (Z−OECD)/total(Z)     signed %
ratio   = lambda Z: np.where(ref, Z / (Zo + 1e-12), np.nan)           # Z / OECD  (×)

p95 = float(np.nanpercentile(adiff(Za), 95)) or 1.0                   # shared abs-diff cap
CAP = 100                                                             # relative-diff cap (%)
cr  = float(np.nanpercentile(np.abs(rownorm(Za)), 95)) or 1.0         # symmetric caps for the
ct  = float(np.nanpercentile(np.abs(totnorm(Za)), 95)) or 1.0         # signed normalised panels

fig, axes = plt.subplots(2, 5, figsize=(34, 14))
for row, (Z, tag) in enumerate([(Zb, "before"), (Za, "after")]):
    draw_hm(axes[row, 0], np.minimum(adiff(Z), p95),
            f"{tag} — |Z−OECD|  (M$, cap p95={p95:,.0f})", "Greens", 0, p95, fmt="M$")
    draw_hm(axes[row, 1], np.minimum(reldp(Z), CAP),
            f"{tag} — |Z−OECD|/OECD  (cap {CAP}%)   div={div(Z, Zo):.3f}", "Reds", 0, CAP)
    draw_hm(axes[row, 2], np.clip(rownorm(Z), -cr, cr),
            f"{tag} — (Z−OECD)/rowSum(Z)  (cap {cr:.1f}%)", "RdBu_r", -cr, cr, vcenter=0.0)
    draw_hm(axes[row, 3], np.clip(totnorm(Z), -ct, ct),
            f"{tag} — (Z−OECD)/totalSum(Z)  (cap {ct:.2f}%)", "RdBu_r", -ct, ct, vcenter=0.0)
    draw_hm(axes[row, 4], np.clip(ratio(Z), 0, 2.5),
            f"{tag} — ratio Z/OECD  (cap 2.5×)", "RdBu_r", 0, 2.5, vcenter=1.0, fmt="x")

fig.suptitle(f"Z[USA × USA] — diff (abs / rel / row-norm / total-norm) and ratio vs OECD ({YEAR})",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / f"Z_diff_norms_ratio_{YEAR}.png", dpi=160, bbox_inches="tight")
plt.show()
